In [2]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal

df = pd.read_csv("../data/processed/screensense_engineered.csv")

df.head()

,record_id,year,month,day_of_week,country,age_group,gender,app_category,app_name,subscription_type,...,sleep_disruption_from_phone,screen_time_concern,mental_health_impact,digital_wellbeing_feature_used,high_screen_time,long_sessions,high_session_frequency,high_app_opens,high_notifications,behavioural_persona
0,APP0000001,2024,11,Thursday,Turkey,45-54,Male,Social Media,Snapchat,Paid Yearly,...,Moderate,Yes,Positive,Yes,0,0,1,1,0,Frequent Checkers
1,APP0000002,2022,5,Friday,USA,25-34,Male,Shopping,Etsy,Freemium,...,Not Reported,Yes,Negative,Yes,0,0,0,0,0,Low-Intensity Users
2,APP0000003,2024,9,Sunday,USA,25-34,Male,Food Delivery,Deliveroo,Free,...,Not Reported,No,Negative,No,0,0,0,0,0,Low-Intensity Users
3,APP0000004,2023,7,Tuesday,Pakistan,25-34,Male,Entertainment/Streaming,Twitch,Freemium,...,Moderate,Somewhat,Negative,Yes,0,1,0,0,0,Long-Session Users
4,APP0000005,2023,3,Tuesday,UAE,25-34,Male,Communication/Messaging,Viber,Freemium,...,Moderate,Yes,Neutral,Yes,0,0,0,0,0,Low-Intensity Users


In [3]:
df.columns.tolist()

['record_id',
 'year',
 'month',
 'day_of_week',
 'country',
 'age_group',
 'gender',
 'app_category',
 'app_name',
 'subscription_type',
 'daily_screen_time_minutes',
 'session_duration_minutes',
 'sessions_per_day',
 'app_opens_per_day',
 'notifications_received_per_day',
 'notification_settings',
 'primary_usage_time',
 'sleep_disruption_from_phone',
 'screen_time_concern',
 'mental_health_impact',
 'digital_wellbeing_feature_used',
 'high_screen_time',
 'long_sessions',
 'high_session_frequency',
 'high_app_opens',
 'high_notifications',
 'behavioural_persona']

# 04 — Statistical Validation

## Objective

The exploratory analysis identified distinct smartphone engagement patterns and behavioural personas.

This notebook statistically evaluates whether behavioural personas differ significantly across key engagement metrics.

Because the engagement variables may not satisfy normality assumptions, the Kruskal–Wallis test is used as a non-parametric alternative to one-way ANOVA.

**Null hypothesis (H₀):** The distribution of an engagement metric is the same across behavioural personas.

**Alternative hypothesis (H₁):** At least one behavioural persona differs significantly.

In [4]:
df['behavioural_persona'].value_counts()

behavioural_persona
Low-Intensity Users           3158
Frequent Checkers             2780
Notification-Exposed Users    1487
Deep Engagement Users          959
High Screen-Time Users         819
Long-Session Users             797
Name: count, dtype: int64

## 1. Persona-Level Engagement Summary

Before hypothesis testing, the median engagement behaviour of each persona is compared across the five core engagement metrics.

In [5]:
engagement_metrics = [
    'daily_screen_time_minutes',
    'session_duration_minutes',
    'sessions_per_day',
    'app_opens_per_day',
    'notifications_received_per_day'
]

persona_summary = (
    df.groupby('behavioural_persona')[engagement_metrics]
      .median()
      .round(2)
)

persona_summary

,daily_screen_time_minutes,session_duration_minutes,sessions_per_day,app_opens_per_day,notifications_received_per_day
behavioural_persona,,,,,
Deep Engagement Users,162.0,65.50,6.0,8.0,25.0
Frequent Checkers,56.0,27.75,8.0,11.0,25.0
High Screen-Time Users,159.0,27.60,5.0,7.0,24.0
Long-Session Users,65.7,59.10,5.0,7.0,24.0
Low-Intensity Users,43.3,23.15,5.0,7.0,23.0
Notification-Exposed Users,57.2,27.20,5.0,7.0,31.0


## 2. Kruskal–Wallis Test

The Kruskal–Wallis H-test evaluates whether the distribution of each engagement metric differs significantly across behavioural personas.

A significance threshold of **α = 0.05** is used. A p-value below 0.05 indicates sufficient evidence to reject the null hypothesis.

In [6]:
results = []

for metric in engagement_metrics:
    groups = [
        group[metric].dropna().values
        for _, group in df.groupby('behavioural_persona')
    ]

    h_stat, p_value = kruskal(*groups)

    results.append({
        'metric': metric,
        'h_statistic': h_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    })

test_results = pd.DataFrame(results)

test_results

,metric,h_statistic,p_value,significant
0,daily_screen_time_minutes,3972.339092,0.0,True
1,session_duration_minutes,3797.941510,0.0,True
2,sessions_per_day,1479.045367,0.0,True
3,app_opens_per_day,2133.505767,0.0,True
4,notifications_received_per_day,3420.747335,0.0,True


### Interpretation

All five engagement metrics show statistically significant differences across the behavioural personas (Kruskal–Wallis, p < 0.05).

However, these results should be interpreted as **structural validation rather than independent evidence of behavioural differences**. The personas were rule-based and derived from these engagement indicators; therefore, separation across the underlying metrics is expected by design.

The results confirm that the segmentation logic produces behaviourally distinct groups and is internally consistent. Independent outcome variables are analysed next to determine whether these engagement patterns are associated with differences in reported digital wellbeing.

## 3. Behavioural Personas and Digital Wellbeing Outcomes

To evaluate whether engagement personas are associated with reported digital wellbeing, Chi-square tests of independence are applied to three categorical outcomes:

- Sleep disruption from phone usage
- Screen-time concern
- Reported mental-health impact

Unlike the engagement metrics used to construct the personas, these wellbeing outcomes were not used in the segmentation logic. They therefore provide an independent basis for evaluating whether behavioural engagement patterns are associated with differences in self-reported wellbeing.

In [7]:
from scipy.stats import chi2_contingency

wellbeing_outcomes = [
    'sleep_disruption_from_phone',
    'screen_time_concern',
    'mental_health_impact'
]

chi_square_results = []

for outcome in wellbeing_outcomes:
    contingency_table = pd.crosstab(
        df['behavioural_persona'],
        df[outcome]
    )

    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    chi_square_results.append({
        'outcome': outcome,
        'chi2_statistic': chi2,
        'degrees_of_freedom': dof,
        'p_value': p_value,
        'significant': p_value < 0.05
    })

chi_square_results = pd.DataFrame(chi_square_results)

chi_square_results

,outcome,chi2_statistic,degrees_of_freedom,p_value,significant
0,sleep_disruption_from_phone,23.071179,15,0.082640,False
1,screen_time_concern,6.193606,10,0.798743,False
2,mental_health_impact,17.112324,20,0.645668,False


### Key Finding

No significant association was found between behavioural personas and sleep disruption, screen-time concern, or mental-health impact (p > 0.05).

Users with very different engagement patterns reported similar wellbeing outcomes. This suggests that high or frequent smartphone usage alone may not reliably indicate perceived digital wellbeing concerns.

In [8]:
def cramers_v(contingency_table):
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.to_numpy().sum()
    rows, cols = contingency_table.shape

    return np.sqrt(chi2 / (n * min(rows - 1, cols - 1)))


association_results = []

for outcome in wellbeing_outcomes:
    table = pd.crosstab(
        df['behavioural_persona'],
        df[outcome]
    )

    association_results.append({
        'outcome': outcome,
        'cramers_v': cramers_v(table)
    })

pd.DataFrame(association_results).round(3)

,outcome,cramers_v
0,sleep_disruption_from_phone,0.028
1,screen_time_concern,0.018
2,mental_health_impact,0.021


## Conclusion

Behavioural personas showed clear differences in smartphone engagement patterns but little association with self-reported wellbeing outcomes.

This reinforces the central finding of the analysis: digital engagement intensity alone does not reliably explain perceived digital wellbeing.